# VoxCPM2 在 ModelScope 免费 GPU（T4/V100）上合成音频 —— turnkey 一键验证

对标已在 **Modal T4** 上验证成功的 `modal_voxcpm2_smoke.py`，本 notebook 在魔搭 GPU 笔记本中完成：
下载模型 → 兼容补丁 → 官方 `voxcpm` 库加载 → `generate()` 合成 → 保存 .wav

**红线#5**：本 notebook 不接受真实凭据。运行前请把下方 `MODELSCOPE_API_TOKEN` 占位替换为你在聊天中获取的令牌
（此值只存在于当前会话内存，**不会**写入任何仓库文件）。建议用后在魔搭控制台轮换（rotate）。


In [ ]:
# ⚠️ 在此输入你的 ModelScope 访问令牌（占位请替换；不落库）
import os
os.environ["MODELSCOPE_API_TOKEN"] = "<REDACTED_MODELSCOPE_TOKEN>"  # ← 替换为你的 ms-... 令牌

# 模型下载/推理环境
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_DISABLE_SSL_VERIFY"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
MODEL_CACHE = os.environ.get("MODEL_CACHE", "/mnt/workspace/VoxCPM2")

print("MODELSCOPE_API_TOKEN 已注入:", "✅" if not os.environ["MODELSCOPE_API_TOKEN"].startswith("<") else "❌ 仍是占位，请替换")
print("MODEL_CACHE =", MODEL_CACHE)


In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory//1024//1024} MB")


In [ ]:
import sys, os, types, subprocess, importlib.util

# ---- 兼容补丁：torch.load weights_only=False & BlockMask ----
import torch

_real = torch.load
def _patched_load(*a, **kw):
    if "weights_only" not in kw:
        kw["weights_only"] = False
    return _real(*a, **kw)
torch.load = _patched_load
sys.modules["torch"].load = _patched_load

if not hasattr(torch.nn, "attention"):
    torch.nn.attention = types.ModuleType("attention")
    sys.modules["torch.nn.attention"] = torch.nn.attention
try:
    import torch.nn.attention.flex_attention as f
except ImportError:
    f = types.ModuleType("flex_attention")
    torch.nn.attention.flex_attention = f
    sys.modules["torch.nn.attention.flex_attention"] = f
if not isinstance(getattr(f, "BlockMask", None), type):
    class _BM: pass
    f.BlockMask = _BM
    sys.modules["torch.nn.attention.flex_attention.BlockMask"] = _BM
print("补丁已应用 (weights_only=False, BlockMask mock)")

# ---- 安装官方推理库 voxcpm + 依赖 ----
for pkg in ["voxcpm==2.0.3", "soundfile", "librosa", "modelscope", "boto3", "redis"]:
    mod = pkg.split("==")[0]
    if importlib.util.find_spec(mod) is None:
        print(f"安装 {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", pkg], check=False)
print("依赖就绪")


In [ ]:
from pathlib import Path
import os
MODEL_DIR = Path(MODEL_CACHE)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

def try_modelscope():
    from modelscope import snapshot_download as ms
    for repo in ("OpenBMB/VoxCPM2", "openbmb/VoxCPM2"):
        try:
            print(f"📥 [ModelScope] {repo} -> {MODEL_DIR}")
            ms(repo, local_dir=str(MODEL_DIR))  # 使用 MODELSCOPE_API_TOKEN 认证
            if (MODEL_DIR / "config.json").exists():
                print("✅ ModelScope 下载成功")
                return True
        except Exception as e:
            print(f"⚠️ {repo}: {str(e)[:120]}")
    return False

def try_hf_mirror():
    import requests
    from huggingface_hub import HfApi
    endpoint = os.environ.get("HF_ENDPOINT", "https://hf-mirror.com")
    repo = "openbmb/VoxCPM2"
    try:
        info = HfApi(endpoint=endpoint).model_info(repo)
        files = sorted(s.rfilename for s in info.siblings)
    except Exception as e:
        print(f"❌ HF 列表失败: {e}"); return False
    ok = 0
    for fname in files:
        dest = MODEL_DIR / fname
        if dest.exists() and dest.stat().st_size > 0:
            ok += 1; continue
        url = f"{endpoint}/{repo}/resolve/main/{fname}"
        try:
            r = requests.get(url, timeout=180, stream=True, allow_redirects=True, verify=False)
            if r.status_code == 200:
                tp = dest.with_suffix(dest.suffix + ".incomplete")
                with open(tp, "wb") as fh:
                    for c in r.iter_content(8192): fh.write(c)
                tp.replace(dest); ok += 1
                print(f"   ✅ {fname} ({dest.stat().st_size/1e6:.2f} MB)")
        except Exception as e:
            print(f"   ❌ {fname}: {e}")
    print(f"[HF镜像] {ok}/{len(files)}")
    return ok == len(files)

if (MODEL_DIR / "config.json").exists() and any(MODEL_DIR.glob("*.safetensors")):
    print(f"✅ 模型已缓存: {MODEL_DIR}")
else:
    if not (try_modelscope() or try_hf_mirror()):
        raise RuntimeError("所有下载方案失败")


In [ ]:
import json, time
# 修复 config.json（补 model_type, 供官方库加载）
cfg = json.loads((MODEL_DIR / "config.json").read_text())
if "model_type" not in cfg:
    cfg["model_type"] = "voxcpm2"
    (MODEL_DIR / "config.json").write_text(json.dumps(cfg, indent=2))
    print("✅ config.json 已修复")

from voxcpm import VoxCPM
t0 = time.time()
model = VoxCPM.from_pretrained(str(MODEL_DIR), load_denoiser=False, optimize=False, device="cuda")
sr = getattr(model.tts_model, "sample_rate", 48000)
print(f"✅ 模型加载完成 {time.time()-t0:.1f}s, sr={sr}")


In [ ]:
import time, numpy as np, soundfile as sf
texts = [
    "你好，这是 VoxCPM2 模型在魔搭 GPU 上生成的中文语音测试。",
    "This is a test of VoxCPM2 text to speech on a ModelScope GPU.",
]
os.makedirs("/mnt/workspace/voxcpm2_out", exist_ok=True)
for i, text in enumerate(texts):
    print(f"\nTest {i+1}: {text[:40]}")
    t0 = time.time()
    try:
        wav = model.generate(text=text, cfg_value=2.0, inference_timesteps=10)
    except TypeError:
        wav = model.generate(target_text=text, max_len=1024, cfg_value=2.0, inference_timesteps=10)
    ta = time.time()
    wav = np.asarray(wav).astype("float32").reshape(-1)
    dur = len(wav) / sr
    out = f"/mnt/workspace/voxcpm2_out/voxcpm2_ms_{i}.wav"
    sf.write(out, wav, sr)
    print(f"   ✅ 时长 {dur:.2f}s | 合成 {ta-t0:.2f}s | RTF {(ta-t0)/dur:.3f} | {out}")
print("\n=== 合成完成, 请在左侧文件区下载 .wav 试听 ===")


In [ ]:
import torch
print("=== 生成的音频 ===")
import os
for f in sorted(os.listdir("/mnt/workspace/voxcpm2_out")):
    if f.endswith(".wav"):
        wav, r = sf.read(f"/mnt/workspace/voxcpm2_out/{f}")
        print(f"  {f}: {len(wav)/r:.2f}s, {r}Hz")
if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated()//1024//1024
    total = torch.cuda.get_device_properties(0).total_memory//1024//1024
    print(f"\n显存: {alloc}/{total} MB")
print("\n如需接入算力池: 注入 REDIS_HOST/R2_* 真值后运行 modelscope_worker.py 即可进入 BLPOP 消费循环")
